# <center>Advanced GroupBy

This is where you stop using GroupBy for simple summaries and start using it for real analytics.\
**The three superpowers are:**\
groupby().transform()\
groupby().filter()\
groupby().apply()\
If you master these three, you can solve most group-wise business problems.

**Dataset**

In [1]:
import pandas as pd
df = pd.DataFrame({
    "Name":["Ali","Sara","Ahmed","Ayesha","Bilal","Usman"],
    "Department":["HR","HR","IT","IT","Finance","Finance"],
    "Salary":[50000,70000,90000,80000,60000,75000],
    "Experience":[2,5,8,6,3,7]
})
df

,Name,Department,Salary,Experience
0,Ali,HR,50000,2
1,Sara,HR,70000,5
2,Ahmed,IT,90000,8
3,Ayesha,IT,80000,6
4,Bilal,Finance,60000,3
5,Usman,Finance,75000,7


**1. transform()**\
Normal Aggreagation:

In [2]:
df.groupby("Department")["Salary"].mean()

Department
Finance    67500.0
HR         60000.0
IT         85000.0
Name: Salary, dtype: float64

transform:

In [3]:
df.groupby("Department")["Salary"].transform("mean")

0    60000.0
1    60000.0
2    85000.0
3    85000.0
4    67500.0
5    67500.0
Name: Salary, dtype: float64

Example 1: Department Average Salary

In [4]:
df["Dept_Avg"] = df.groupby("Department")["Salary"].transform("mean")
df

,Name,Department,Salary,Experience,Dept_Avg
0,Ali,HR,50000,2,60000.0
1,Sara,HR,70000,5,60000.0
2,Ahmed,IT,90000,8,85000.0
3,Ayesha,IT,80000,6,85000.0
4,Bilal,Finance,60000,3,67500.0
5,Usman,Finance,75000,7,67500.0


Example 2: Salary Difference from Department Average\
Industry question: Is this employee above or below Department's average salary?

In [6]:
import numpy as np
df["Sal_diff"] = np.where(
    df["Salary"]>df["Dept_Avg"],
    "Above Average",
    "Below Average"
)
df

,Name,Department,Salary,Experience,Dept_Avg,Sal_diff
0,Ali,HR,50000,2,60000.0,Below Average
1,Sara,HR,70000,5,60000.0,Above Average
2,Ahmed,IT,90000,8,85000.0,Above Average
3,Ayesha,IT,80000,6,85000.0,Below Average
4,Bilal,Finance,60000,3,67500.0,Below Average
5,Usman,Finance,75000,7,67500.0,Above Average


Example 3: Group-wise Normalization

In [8]:
df["Salary-z"] = df.groupby("Department")["Salary"].transform(
    lambda x:
    (x - x.mean()) / x.std()
)

In [9]:
df

,Name,Department,Salary,Experience,Dept_Avg,Sal_diff,Salary-z
0,Ali,HR,50000,2,60000.0,Below Average,-0.707107
1,Sara,HR,70000,5,60000.0,Above Average,0.707107
2,Ahmed,IT,90000,8,85000.0,Above Average,0.707107
3,Ayesha,IT,80000,6,85000.0,Below Average,-0.707107
4,Bilal,Finance,60000,3,67500.0,Below Average,-0.707107
5,Usman,Finance,75000,7,67500.0,Above Average,0.707107


Example 4: Rankings Within Groups\
Who earns the most inside each department?

In [10]:
df["Rank"] = df.groupby("Department")["Salary"].rank(ascending=False)
df

,Name,Department,Salary,Experience,Dept_Avg,Sal_diff,Salary-z,Rank
0,Ali,HR,50000,2,60000.0,Below Average,-0.707107,2.0
1,Sara,HR,70000,5,60000.0,Above Average,0.707107,1.0
2,Ahmed,IT,90000,8,85000.0,Above Average,0.707107,1.0
3,Ayesha,IT,80000,6,85000.0,Below Average,-0.707107,2.0
4,Bilal,Finance,60000,3,67500.0,Below Average,-0.707107,2.0
5,Usman,Finance,75000,7,67500.0,Above Average,0.707107,1.0


**2. filter()**\
filter keeps or removes entire groups.\
Group --> Condition --> Keep? or Remove?

Industry Question: Keep departments whose average salary exceeds 70000

In [12]:
filtered = df.groupby("Department").filter(
    lambda x:
    x["Salary"].mean() > 70000
)
filtered

,Name,Department,Salary,Experience,Dept_Avg,Sal_diff,Salary-z,Rank
2,Ahmed,IT,90000,8,85000.0,Above Average,0.707107,1.0
3,Ayesha,IT,80000,6,85000.0,Below Average,-0.707107,2.0


Example 2: Keep Large Departments\
Keep departments with at least 3 employees.

In [13]:
large = df.groupby("Department").filter(
    lambda x:
    len(x) >= 3
)
large

,Name,Department,Salary,Experience,Dept_Avg,Sal_diff,Salary-z,Rank


**Industry Examples:**\
Remove tiny customer segments\
Remove rare categories\
Remove cities with insufficient data

**3. apply()**\
agg() --> Simple summaries\
transform() --> Same-size output\
apply() --> ANYTHING\
**Think:** Custom Group Processing

Example 1: Salary Range per Department

In [15]:
def salary_range(group):
    return group.max() - group.min()
result = df.groupby("Department")["Salary"].apply(salary_range)
result

Department
Finance    15000
HR         20000
IT         10000
Name: Salary, dtype: int64

Example 2: Top Earner in every Department

In [16]:
def top_employee(group):
    return group.nlargest(1)
result = df.groupby("Department")["Salary"].apply(top_employee)
result

Department   
Finance     5    75000
HR          1    70000
IT          2    90000
Name: Salary, dtype: int64